Undersampling + Residual BLock + Sliding Window CNN + LSTM (no Attention)

In [1]:
import pandas as pd
train_df = pd.read_parquet(r"final_data_http\chunk-based-split-3\train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"final_data_http\chunk-based-split-3\valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"final_data_http\chunk-based-split-3\test_df_prepared.parquet", engine="pyarrow")
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

train_df.sort_values(by='timestamp_num', inplace=True)
valid_df.sort_values(by='timestamp_num', inplace=True)
test_df.sort_values(by='timestamp_num', inplace=True)

cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [2]:
import torch
from torch.utils.data import Dataset
import numpy as np

class UndersampledSlidingWindowDataset(Dataset):
    def __init__(self, df, time_steps, max_samples_per_class=20000, step=5):
        self.X = torch.tensor(df.drop(columns=['label']).values, dtype=torch.float32)
        self.y = torch.tensor(df['label'].values, dtype=torch.long)
        self.time_steps = time_steps
        self.step = step
        
        print(f"Đang tính toán các cửa sổ hợp lệ (global step={step}) và Undersampling...")
        
        # tạo mảng index cho step = 1 (dành riêng cho các class hiếm cần bảo toàn)
        all_indices_step1 = np.arange(0, len(self.X) - self.time_steps + 1, 1)
        window_labels_step1 = self.y[all_indices_step1 + self.time_steps - 1].numpy()
        
        # tạo mảng index theo step được truyền vào (cho các class còn lại)
        all_indices_stepped = np.arange(0, len(self.X) - self.time_steps + 1, self.step)
        window_labels_stepped = self.y[all_indices_stepped + self.time_steps - 1].numpy()
        
        self.valid_indices = []
        
        # lặp qua từng class (Lấy từ step=1 để đảm bảo không sót class nào)
        classes = np.unique(window_labels_step1)
        rng = np.random.default_rng(seed=42)
        for c in classes:
            # ƯU TIÊN GIỮ NGUYÊN băm cửa sổ 1-1 cho Class 2 và 6
            if c in [100]:
                c_indices = all_indices_step1[window_labels_step1 == c]
            else:
                c_indices = all_indices_stepped[window_labels_stepped == c]
                
            count = len(c_indices)
            
            # nếu class đó có nhiều mẫu hơn ngưỡng max_samples_per_class
            if count > max_samples_per_class:
                # chọn ngẫu nhiên không hoàn lại
                 
                c_indices = rng.choice(c_indices, max_samples_per_class, replace=False)
                print(f"Class {c}: Giảm từ {count} xuống {max_samples_per_class} cửa sổ.")
            else:
                # nếu ít hơn thì giữ nguyên
                if c in [2, 6]:
                    print(f"Class {c}: Giữ nguyên {count} cửa sổ (sử dụng step=1 để bảo toàn).")
                else:
                    print(f"Class {c}: Giữ nguyên {count} cửa sổ (step={self.step}).")
                
            self.valid_indices.extend(c_indices)
            
        # Trộn đều danh sách index để các batch đa dạng hơn
        rng.shuffle(self.valid_indices)
        self.valid_indices = np.array(self.valid_indices) # Chuyển sang ndarray
        print(f"Tổng số cửa sổ sau khi lọc và Undersampling: {len(self.valid_indices)}")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        # lấy các index đã được lọc và xáo trộn
        actual_idx = self.valid_indices[idx]
        
        # lấy feature và label của cửa sổ trượt tại index thực sự
        window_X = self.X[actual_idx : actual_idx + self.time_steps]
        label_y = self.y[actual_idx + self.time_steps - 1]
        
        return window_X, label_y

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

MAX_SAMPLES = 20000 
TIME_STEPS = 10
BATCH_SIZE = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TEST_STEP_SIZE = 1
TRAIN_STEP_SIZE = 5 
NUM_FEATURES = train_df.shape[1] - 1
NUM_CLASSES = len(train_df['label'].unique())

# tối đa mỗi class sẽ có 20000 của sổ trượt
print(f"Khởi tạo tập Train (có Undersampling, set Train Step = {TRAIN_STEP_SIZE})...")
train_dataset = UndersampledSlidingWindowDataset(train_df, TIME_STEPS, max_samples_per_class=MAX_SAMPLES, step=TRAIN_STEP_SIZE)

print(f"Khởi tạo tập Val và Test (Không Undersampling, giữ nguyên gốc, set Test Step = {TEST_STEP_SIZE})...")
# để max_samples_per_class cho tập val và test lớn để không xóa cửa sổ trượt nào
val_dataset = UndersampledSlidingWindowDataset(valid_df, TIME_STEPS, max_samples_per_class=10000000, step=1) 
test_dataset = UndersampledSlidingWindowDataset(test_df, TIME_STEPS, max_samples_per_class=10000000, step=1)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Khởi tạo tập Train (có Undersampling, set Train Step = 5)...
Đang tính toán các cửa sổ hợp lệ (global step=5) và Undersampling...
Class 0: Giữ nguyên 12897 cửa sổ (step=5).
Class 1: Giảm từ 314644 xuống 20000 cửa sổ.
Class 2: Giữ nguyên 1633 cửa sổ (sử dụng step=1 để bảo toàn).
Class 3: Giảm từ 23563 xuống 20000 cửa sổ.
Class 4: Giữ nguyên 12333 cửa sổ (step=5).
Class 5: Giữ nguyên 1592 cửa sổ (step=5).
Class 6: Giữ nguyên 7698 cửa sổ (sử dụng step=1 để bảo toàn).
Class 7: Giảm từ 69830 xuống 20000 cửa sổ.
Class 8: Giữ nguyên 10884 cửa sổ (step=5).
Class 9: Giảm từ 26987 xuống 20000 cửa sổ.
Class 10: Giữ nguyên 12065 cửa sổ (step=5).
Tổng số cửa sổ sau khi lọc và Undersampling: 139102
Khởi tạo tập Val và Test (Không Undersampling, giữ nguyên gốc, set Test Step = 1)...
Đang tính toán các cửa sổ hợp lệ (global step=1) và Undersampling...
Class 0: Giữ nguyên 14880 cửa sổ (step=1).
Class 1: Giữ nguyên 363049 cửa sổ (step=1).
Class 2: Giữ nguyên 1884 cửa sổ (sử dụng step=1 để bảo toàn).
Cla

In [4]:
import torch
import torch.nn as nn

# VŨ KHÍ MỚI: Squeeze-and-Excitation (SE Block) - Chống Drift Kênh Đặc Trưng
class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super(SEBlock1D, self).__init__()
        # Global Average Pooling theo chiều thời gian
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        # Bộ tạo trọng số động
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c) # "Ngửi" toàn bộ cửa sổ để đánh giá độ tin cậy của từng kênh
        y = self.fc(y).view(b, c, 1)    # Tính ra thang điểm 0-1 cho từng kênh
        return x * y.expand_as(x)       # Bóp nghẹt kênh bị nhiễm Drift, Khuếch đại kênh chuẩn

# KHỐI RESIDUAL KẾT HỢP SE-BLOCK
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super(ResidualBlock1D, self).__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)
        self.gn1 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding)
        self.gn2 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.dropout = nn.Dropout1d(0.2)
        
        # Nhúng cơ chế SE (Channel Attention)
        self.se = SEBlock1D(out_channels)
        
        # Đường tắt (Shortcut Connection)
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1),
                nn.GroupNorm(num_groups=8, num_channels=out_channels)
            )
            
    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv1(x)
        out = self.gn1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.conv2(out)
        out = self.gn2(out)
        
        # CHỐT CHẶN SE-BLOCK: Lọc bỏ tín hiệu rác do Concept Drift gây ra trên các biến (Features) mỏng manh
        out = self.se(out)
        
        out += residual  
        out = self.relu(out)
        return out


# MODEL CNN-BiLSTM (ĐÃ LƯỢC BỎ ATTENTION)
class CNN_BiLSTM_NoAttention(nn.Module):
    def __init__(self, num_features, num_classes, time_steps, hidden_size=128):
        super(CNN_BiLSTM_NoAttention, self).__init__()
        
        # Thay thế CNN thường bằng Khối Residual Tích hợp SE
        self.res1 = ResidualBlock1D(num_features, 64)
        self.res2 = ResidualBlock1D(64, 128)
        
        # Giảm chiều dài chuỗi đi một nửa
        self.pool = nn.MaxPool1d(kernel_size=2) 
        
        self.bilstm = nn.LSTM(input_size=128, hidden_size=hidden_size, 
                              batch_first=True, bidirectional=True)
        
        # LayerNorm (Chuẩn hóa biên độ)
        self.layer_norm = nn.LayerNorm(hidden_size * 2)
        
        # Đã loại bỏ self.attention
        
        self.dropout = nn.Dropout(0.5)
        
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        # Tăng cường ổn định ở bộ phân loại cuối
        self.fc_ln = nn.LayerNorm(64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, num_classes)
        
    def forward(self, x):
        # x: (Batch, SeqLen, Features)
        x = x.permute(0, 2, 1) 
        
        # Đi qua các khối Residual CNN + SE Block
        x = self.res1(x)
        x = self.res2(x)
        
        x = self.pool(x)
        
        x = x.permute(0, 2, 1)
        
        out, _ = self.bilstm(x)
        
        # CHUẨN HOÁ LAYER NORM THEO TỪNG TIME-STEP
        out = self.layer_norm(out)
        
        # THAY THẾ ATTENTION: Sử dụng Global Average Pooling theo chiều sequence
        # Lấy trung bình tất cả các time-steps để nén về (Batch, hidden_size * 2)
        context_vector = torch.mean(out, dim=1)
        
        out = self.dropout(context_vector)
        out = self.fc1(out)
        out = self.fc_ln(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        
        return out

In [5]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score

model = CNN_BiLSTM_NoAttention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
# Dùng Cross Entropy với Label Smoothing = 0.1 để chống over-confidence thay vì Focal Loss (hạn chế kìm nghẹt F1 các class có Concept Drift)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 10

for epoch in range(EPOCHS):
    
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping để cắt tỉa các vi phân nảy loạn xạ khi model học các mẫu Hard bị drift
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_1.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/30] - Train Loss: 0.9405, Train Macro F1: 0.8065 | Val Loss: 0.6206, Val Macro F1: 0.8893


Epoch [2/30] - Train Loss: 0.7249, Train Macro F1: 0.9353 | Val Loss: 0.6105, Val Macro F1: 0.8982


Epoch [3/30] - Train Loss: 0.7008, Train Macro F1: 0.9495 | Val Loss: 0.6065, Val Macro F1: 0.9090


Epoch [4/30] - Train Loss: 0.6852, Train Macro F1: 0.9579 | Val Loss: 0.6143, Val Macro F1: 0.8787


...

Epoch [19/30] - Train Loss: 0.6172, Train Macro F1: 0.9870 | Val Loss: 0.5872, Val Macro F1: 0.9360


Epoch [20/30] - Train Loss: 0.6156, Train Macro F1: 0.9867 | Val Loss: 0.5874, Val Macro F1: 0.9317


Epoch [21/30] - Train Loss: 0.6131, Train Macro F1: 0.9880 | Val Loss: 0.5913, Val Macro F1: 0.9379


Epoch [22/30] - Train Loss: 0.6005, Train Macro F1: 0.9926 | Val Loss: 0.5874, Val Macro F1: 0.9424

[Early Stopping] Model không cải thiện sau 10 epochs. Dừng huấn luyện.

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---


              precision    recall  f1-score   support

           0     0.8314    0.7872    0.8087     19846
           1     0.9972    1.0000    0.9986    484077
           2     0.3029    0.9956    0.4644      2515
           3     0.9931    0.8368    0.9083     36253
           4     0.5969    0.8221    0.6916     18979
           5     0.9984    0.9914    0.9949      2451
           6     0.7498    0.7048    0.7266     11847
           7     1.0000    0.9999    0.9999    107436
           8     0.6905    0.9784    0.8096     16746
           9     0.9996    0.6436    0.7830     41514
          10     0.9354    0.9880    0.9610     18567

    accuracy                         0.9573    760231
   macro avg     0.8268    0.8862    0.8315    760231
weighted avg     0.9688    0.9573    0.9590    760231



UnderSampling + Residual Block + CNN + LSTM + Attention (no Sliding Window)

In [6]:
import pandas as pd
train_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\test_df_prepared.parquet", engine="pyarrow")
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

train_df.sort_values(by='timestamp_num', inplace=True)
valid_df.sort_values(by='timestamp_num', inplace=True)
test_df.sort_values(by='timestamp_num', inplace=True)

cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
NUM_FEATURES = train_dataset.X.shape[1]

# CƠ CHẾ ATTENTION THEO THỜI GIAN (Temporal Attention)
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs):
        scores = self.attention(lstm_outputs)
        weights = torch.softmax(scores, dim=1)
        context_vector = torch.sum(weights * lstm_outputs, dim=1)
        return context_vector, weights

# VŨ KHÍ MỚI: Squeeze-and-Excitation (SE Block) - Chống Drift Kênh Đặc Trưng
class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super(SEBlock1D, self).__init__()
        # Global Average Pooling theo chiều thời gian
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        # Bộ tạo trọng số động
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c) # "Ngửi" toàn bộ cửa sổ để đánh giá độ tin cậy của từng kênh
        y = self.fc(y).view(b, c, 1)    # Tính ra thang điểm 0-1 cho từng kênh
        return x * y.expand_as(x)       # Bóp nghẹt kênh bị nhiễm Drift, Khuếch đại kênh chuẩn

# KHỐI RESIDUAL KẾT HỢP SE-BLOCK
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super(ResidualBlock1D, self).__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)
        self.gn1 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding)
        self.gn2 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.dropout = nn.Dropout1d(0.2)
        
        # Nhúng cơ chế SE (Channel Attention)
        self.se = SEBlock1D(out_channels)
        
        # Đường tắt (Shortcut Connection)
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1),
                nn.GroupNorm(num_groups=8, num_channels=out_channels)
            )
            
    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv1(x)
        out = self.gn1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.conv2(out)
        out = self.gn2(out)
        
        # CHỐT CHẶN SE-BLOCK: Lọc bỏ tín hiệu rác do Concept Drift gây ra trên các biến (Features) mỏng manh
        out = self.se(out)
        
        out += residual  
        out = self.relu(out)
        return out


# ĐẠI TU MODEL CNN-BiLSTM
class CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes, time_steps, hidden_size=128):
        super(CNN_BiLSTM_Attention, self).__init__()
        
        # Thay thế CNN thường bằng Khối Residual Tích hợp SE
        self.res1 = ResidualBlock1D(num_features, 64)
        self.res2 = ResidualBlock1D(64, 128)
        
        # Giảm chiều dài chuỗi đi một nửa
        self.pool = nn.MaxPool1d(kernel_size=2) 
        
        self.bilstm = nn.LSTM(input_size=128, hidden_size=hidden_size, 
                              batch_first=True, bidirectional=True)
        
        # LayerNorm (Chuẩn hóa biên độ)
        self.layer_norm = nn.LayerNorm(hidden_size * 2)
        
        self.attention = Attention(hidden_size * 2)
        self.dropout = nn.Dropout(0.5)
        
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        # Tăng cường ổn định ở bộ phân loại cuối
        self.fc_ln = nn.LayerNorm(64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, num_classes)
        
    def forward(self, x):
        # x: (Batch, SeqLen, Features)
        x = x.permute(0, 2, 1) 
        
        # Đi qua các khối Residual CNN + SE Block
        x = self.res1(x)
        x = self.res2(x)
        if x.size(-1) >=2:
            x = self.pool(x)
        
        x = x.permute(0, 2, 1)
        
        out, _ = self.bilstm(x)
        
        # CHUẨN HOÁ LAYER NORM THEO TỪNG TIME-STEP
        out = self.layer_norm(out)
        
        context_vector, attn_weights = self.attention(out)
        
        out = self.dropout(context_vector)
        out = self.fc1(out)
        out = self.fc_ln(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        
        return out


In [8]:
import torch
from torch.utils.data import Dataset
import numpy as np

class UndersampledDataset(Dataset):
    def __init__(self, df, max_samples_per_class=20000):
        # Trích xuất Features và Labels
        self.X = torch.tensor(df.drop(columns=['label']).values, dtype=torch.float32)
        self.y = torch.tensor(df['label'].values, dtype=torch.long)
        
        print(f"Đang tính toán các mẫu hợp lệ và Undersampling...")
        
        # Mảng chứa tất cả index từ 0 đến len(df) - 1
        all_indices = np.arange(len(self.X))
        all_labels = self.y.numpy()
        
        self.valid_indices = []
        
        # Lặp qua từng class
        classes = np.unique(all_labels)
        rng = np.random.default_rng(seed=42)
        
        for c in classes:
            # Lấy tất cả index của các mẫu thuộc class c
            c_indices = all_indices[all_labels == c]
            count = len(c_indices)
            
            # Nếu class đó có nhiều mẫu hơn ngưỡng max_samples_per_class
            if count > max_samples_per_class:
                # Chọn ngẫu nhiên không hoàn lại
                c_indices = rng.choice(c_indices, max_samples_per_class, replace=False)
                print(f"Class {c}: Giảm từ {count} xuống {max_samples_per_class} mẫu.")
            else:
                # Nếu ít hơn hoặc bằng ngưỡng thì giữ nguyên
                print(f"Class {c}: Giữ nguyên {count} mẫu.")
                
            self.valid_indices.extend(c_indices)
            
        # Trộn đều danh sách index để các batch đa dạng hơn khi train
        rng.shuffle(self.valid_indices)
        self.valid_indices = np.array(self.valid_indices) # Chuyển sang ndarray
        print(f"Tổng số mẫu sau khi lọc và Undersampling: {len(self.valid_indices)}")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        # Lấy index thực sự đã được lọc và xáo trộn
        actual_idx = self.valid_indices[idx]
        
        # Lấy feature và label tại dòng (index) tương ứng
        sample_X = self.X[actual_idx].unsqueeze(0) 
        label_y = self.y[actual_idx]
        
        return sample_X, label_y

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

MAX_SAMPLES = 20000 

BATCH_SIZE = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_FEATURES = train_df.shape[1] - 1
NUM_CLASSES = len(train_df['label'].unique())

# tối đa mỗi class sẽ có 20000 của sổ trượt
print(f"Khởi tạo tập Train (có Undersampling, đưa mỗi sample riêng lẻ).")
train_dataset = UndersampledDataset(train_df, max_samples_per_class=MAX_SAMPLES)


print(f"Khởi tạo tập Val và Test (Không Undersampling, đưa mỗi sample riêng lẻ, giữ nguyên gốc)...")
# để max_samples_per_class cho tập val và test lớn để không xóa cửa sổ trượt nào
val_dataset = UndersampledDataset(valid_df, max_samples_per_class=10000000)
test_dataset = UndersampledDataset(test_df, max_samples_per_class=10000000)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Khởi tạo tập Train (có Undersampling, đưa mỗi sample riêng lẻ).
Đang tính toán các mẫu hợp lệ và Undersampling...
Class 0: Giảm từ 64488 xuống 20000 mẫu.
Class 1: Giảm từ 1573220 xuống 20000 mẫu.
Class 2: Giữ nguyên 8164 mẫu.
Class 3: Giảm từ 117813 xuống 20000 mẫu.
Class 4: Giảm từ 61666 xuống 20000 mẫu.
Class 5: Giữ nguyên 7956 mẫu.
Class 6: Giảm từ 38490 xuống 20000 mẫu.
Class 7: Giảm từ 349153 xuống 20000 mẫu.
Class 8: Giảm từ 54420 xuống 20000 mẫu.
Class 9: Giảm từ 134940 xuống 20000 mẫu.
Class 10: Giảm từ 60328 xuống 20000 mẫu.
Tổng số mẫu sau khi lọc và Undersampling: 196120
Khởi tạo tập Val và Test (Không Undersampling, đưa mỗi sample riêng lẻ, giữ nguyên gốc)...
Đang tính toán các mẫu hợp lệ và Undersampling...
Class 0: Giữ nguyên 14880 mẫu.
Class 1: Giữ nguyên 363049 mẫu.
Class 2: Giữ nguyên 1884 mẫu.
Class 3: Giữ nguyên 27186 mẫu.
Class 4: Giữ nguyên 14229 mẫu.
Class 5: Giữ nguyên 1836 mẫu.
Class 6: Giữ nguyên 8880 mẫu.
Class 7: Giữ nguyên 80572 mẫu.
Class 8: Giữ nguyên 1255

In [10]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score

model = CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
# Dùng Cross Entropy với Label Smoothing = 0.1 để chống over-confidence thay vì Focal Loss (hạn chế kìm nghẹt F1 các class có Concept Drift)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 10

for epoch in range(EPOCHS):
    
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping để cắt tỉa các vi phân nảy loạn xạ khi model học các mẫu Hard bị drift
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_1.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/30] - Train Loss: 1.2204, Train Macro F1: 0.6985 | Val Loss: 0.6929, Val Macro F1: 0.7338


Epoch [2/30] - Train Loss: 1.0633, Train Macro F1: 0.7646 | Val Loss: 0.6703, Val Macro F1: 0.7585


Epoch [3/30] - Train Loss: 1.0149, Train Macro F1: 0.7880 | Val Loss: 0.6810, Val Macro F1: 0.7502


Epoch [4/30] - Train Loss: 1.0021, Train Macro F1: 0.7953 | Val Loss: 0.6684, Val Macro F1: 0.7544


...

Epoch [27/30] - Train Loss: 0.8996, Train Macro F1: 0.8409 | Val Loss: 0.6519, Val Macro F1: 0.7835


Epoch [28/30] - Train Loss: 0.9014, Train Macro F1: 0.8408 | Val Loss: 0.6493, Val Macro F1: 0.7809


Epoch [29/30] - Train Loss: 0.9024, Train Macro F1: 0.8415 | Val Loss: 0.6533, Val Macro F1: 0.7755


Epoch [30/30] - Train Loss: 0.9009, Train Macro F1: 0.8424 | Val Loss: 0.6531, Val Macro F1: 0.7814

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---


              precision    recall  f1-score   support

           0     0.6092    0.2376    0.3418     19846
           1     0.9992    1.0000    0.9996    484077
           2     0.5866    0.7932    0.6744      2515
           3     0.9600    0.9108    0.9348     36253
           4     0.5972    0.4347    0.5031     18979
           5     0.9819    0.9967    0.9893      2451
           6     0.5585    0.7089    0.6247     11847
           7     1.0000    0.9898    0.9948    107436
           8     0.4368    0.9008    0.5883     16746
           9     0.9999    0.6919    0.8178     41523
          10     0.5336    0.8419    0.6532     18567

    accuracy                         0.9322    760240
   macro avg     0.7512    0.7733    0.7384    760240
weighted avg     0.9452    0.9322    0.9319    760240



Dynamic Undersampling + Residual Block + CNN + LSTM + Attention + Sliding Window

In [11]:
import pandas as pd
train_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\test_df_prepared.parquet", engine="pyarrow")
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

train_df.sort_values(by='timestamp_num', inplace=True)
valid_df.sort_values(by='timestamp_num', inplace=True)
test_df.sort_values(by='timestamp_num', inplace=True)

cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F 
NUM_FEATURES = train_dataset.X.shape[1]

# CƠ CHẾ ATTENTION THEO THỜI GIAN (Temporal Attention)
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs):
        scores = self.attention(lstm_outputs)
        weights = torch.softmax(scores, dim=1)
        context_vector = torch.sum(weights * lstm_outputs, dim=1)
        return context_vector, weights

# VŨ KHÍ MỚI: Squeeze-and-Excitation (SE Block) - Chống Drift Kênh Đặc Trưng
class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super(SEBlock1D, self).__init__()
        # Global Average Pooling theo chiều thời gian
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        # Bộ tạo trọng số động
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c) # "Ngửi" toàn bộ cửa sổ để đánh giá độ tin cậy của từng kênh
        y = self.fc(y).view(b, c, 1)    # Tính ra thang điểm 0-1 cho từng kênh
        return x * y.expand_as(x)       # Bóp nghẹt kênh bị nhiễm Drift, Khuếch đại kênh chuẩn

# KHỐI RESIDUAL KẾT HỢP SE-BLOCK
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super(ResidualBlock1D, self).__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)
        self.gn1 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding)
        self.gn2 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.dropout = nn.Dropout1d(0.2)
        
        # Nhúng cơ chế SE (Channel Attention)
        self.se = SEBlock1D(out_channels)
        
        # Đường tắt (Shortcut Connection)
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1),
                nn.GroupNorm(num_groups=8, num_channels=out_channels)
            )
            
    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv1(x)
        out = self.gn1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.conv2(out)
        out = self.gn2(out)
        
        # CHỐT CHẶN SE-BLOCK: Lọc bỏ tín hiệu rác do Concept Drift gây ra trên các biến (Features) mỏng manh
        out = self.se(out)
        
        out += residual  
        out = self.relu(out)
        return out


# ĐẠI TU MODEL CNN-BiLSTM
class CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes, time_steps, hidden_size=128):
        super(CNN_BiLSTM_Attention, self).__init__()
        
        # Thay thế CNN thường bằng Khối Residual Tích hợp SE
        self.res1 = ResidualBlock1D(num_features, 64)
        self.res2 = ResidualBlock1D(64, 128)
        
        # Giảm chiều dài chuỗi đi một nửa
        self.pool = nn.MaxPool1d(kernel_size=2) 
        
        self.bilstm = nn.LSTM(input_size=128, hidden_size=hidden_size, 
                              batch_first=True, bidirectional=True)
        
        # LayerNorm (Chuẩn hóa biên độ)
        self.layer_norm = nn.LayerNorm(hidden_size * 2)
        
        self.attention = Attention(hidden_size * 2)
        self.dropout = nn.Dropout(0.5)
        
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        # Tăng cường ổn định ở bộ phân loại cuối
        self.fc_ln = nn.LayerNorm(64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, num_classes)
        
    def forward(self, x):
        # x: (Batch, SeqLen, Features)
        x = x.permute(0, 2, 1) 
        
        # Đi qua các khối Residual CNN + SE Block
        x = self.res1(x)
        x = self.res2(x)
        
        x = self.pool(x)
        
        x = x.permute(0, 2, 1)
        
        out, _ = self.bilstm(x)
        
        # CHUẨN HOÁ LAYER NORM THEO TỪNG TIME-STEP
        out = self.layer_norm(out)
        
        context_vector, attn_weights = self.attention(out)
        
        out = self.dropout(context_vector)
        out = self.fc1(out)
        out = self.fc_ln(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        
        return out


In [13]:
class DynamicUndersampledSlidingWindowDataset(Dataset):
    def __init__(self, df, time_steps, max_samples_per_class=20000, 
                 step=5, resample_each_epoch=False, rare_classes=None):
        

        if rare_classes is None:
            rare_classes = [0, 2, 3, 4, 5, 8, 10]
            
        feat_cols = [col for col in df.columns if col != 'label']
        self.X = torch.as_tensor(df[feat_cols].to_numpy(dtype=np.float32))
        self.y = torch.as_tensor(df['label'].to_numpy(dtype=np.int64))
        
        self.time_steps = time_steps
        self.step = step
        self.max_samples_per_class = max_samples_per_class
        self.resample_each_epoch = resample_each_epoch
        
        # Tạo 2 mảng index: 1 cái step=1 (bảo toàn), 1 cái có step (băm mỏng)
        all_indices_step1 = np.arange(0, len(self.X) - self.time_steps + 1, 1)
        window_labels_step1 = self.y[all_indices_step1 + self.time_steps - 1].numpy()
        
        all_indices_stepped = np.arange(0, len(self.X) - self.time_steps + 1, self.step)
        window_labels_stepped = self.y[all_indices_stepped + self.time_steps - 1].numpy()
        
        self.class_indices = {}
        for c in np.unique(window_labels_step1):
            if c in rare_classes:
                # Bảo toàn 100% dữ liệu cho class hiếm
                self.class_indices[c] = all_indices_step1[window_labels_step1 == c]
            else:
                # Băm mỏng theo step đối với các class đa số
                self.class_indices[c] = all_indices_stepped[window_labels_stepped == c]
            
            print(f"Class {c}: Có sẵn {len(self.class_indices[c])} cửa sổ trong Pool")
            
        self._resample()
    
    def _resample(self):
        self.valid_indices = []
        for c, c_indices in self.class_indices.items():
            count = len(c_indices)
            
            # NẾU LÀ TẬP TRAIN (có bật resample_each_epoch)
            if self.resample_each_epoch:
                if count > self.max_samples_per_class:
                    # Dư thì băm đi (Undersampling)
                    sampled = np.random.choice(c_indices, self.max_samples_per_class, replace=False)
                else:
                    # Thiếu thì nhân bản (Oversampling)
                    sampled = np.random.choice(c_indices, self.max_samples_per_class, replace=True)
                    
            # NẾU LÀ TẬP VAL/TEST (Giữ nguyên gốc 100%, không thêm không bớt)
            else:
                sampled = c_indices
                
            self.valid_indices.extend(sampled)
            
        np.random.shuffle(self.valid_indices)
        self.valid_indices = np.array(self.valid_indices)
    def on_epoch_start(self):
        if self.resample_each_epoch:
            self._resample()

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        actual_idx = self.valid_indices[idx]
        window_X = self.X[actual_idx : actual_idx + self.time_steps]
        label_y = self.y[actual_idx + self.time_steps - 1]
        return window_X, label_y

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

MAX_SAMPLES = 20000 
TIME_STEPS = 10
BATCH_SIZE = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TEST_STEP_SIZE = 1
TRAIN_STEP_SIZE = 5 
NUM_FEATURES = train_df.shape[1] - 1
NUM_CLASSES = len(train_df['label'].unique())

# tối đa mỗi class sẽ có 20000 của sổ trượt
print(f"Khởi tạo tập Train (có Dynamicsampling, set Train Step = {TRAIN_STEP_SIZE})...")
train_dataset = DynamicUndersampledSlidingWindowDataset(train_df, TIME_STEPS, max_samples_per_class=MAX_SAMPLES, step=TRAIN_STEP_SIZE, resample_each_epoch=True)

print(f"Khởi tạo tập Val và Test (Không Undersampling, giữ nguyên gốc, set Test Step = {TEST_STEP_SIZE})...")
# để max_samples_per_class cho tập val và test lớn để không xóa cửa sổ trượt nào
val_dataset = UndersampledSlidingWindowDataset(valid_df, TIME_STEPS, max_samples_per_class=10000000, step=1) 
test_dataset = UndersampledSlidingWindowDataset(test_df, TIME_STEPS, max_samples_per_class=10000000, step=1)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Khởi tạo tập Train (có Dynamicsampling, set Train Step = 5)...
Class 0: Có sẵn 64488 cửa sổ trong Pool
Class 1: Có sẵn 314644 cửa sổ trong Pool
Class 2: Có sẵn 8164 cửa sổ trong Pool
Class 3: Có sẵn 117813 cửa sổ trong Pool
Class 4: Có sẵn 61666 cửa sổ trong Pool
Class 5: Có sẵn 7956 cửa sổ trong Pool
Class 6: Có sẵn 7698 cửa sổ trong Pool
Class 7: Có sẵn 69830 cửa sổ trong Pool
Class 8: Có sẵn 54420 cửa sổ trong Pool
Class 9: Có sẵn 26987 cửa sổ trong Pool
Class 10: Có sẵn 60328 cửa sổ trong Pool
Khởi tạo tập Val và Test (Không Undersampling, giữ nguyên gốc, set Test Step = 1)...
Đang tính toán các cửa sổ hợp lệ (global step=1) và Undersampling...
Class 0: Giữ nguyên 14880 cửa sổ (step=1).
Class 1: Giữ nguyên 363049 cửa sổ (step=1).
Class 2: Giữ nguyên 1884 cửa sổ (sử dụng step=1 để bảo toàn).
Class 3: Giữ nguyên 27186 cửa sổ (step=1).
Class 4: Giữ nguyên 14229 cửa sổ (step=1).
Class 5: Giữ nguyên 1836 cửa sổ (step=1).
Class 6: Giữ nguyên 8880 cửa sổ (sử dụng step=1 để bảo toàn).
Clas

In [15]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score

model = CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
# Dùng Cross Entropy với Label Smoothing = 0.1 để chống over-confidence thay vì Focal Loss (hạn chế kìm nghẹt F1 các class có Concept Drift)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 10

for epoch in range(EPOCHS):
    train_dataset.on_epoch_start() 
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping để cắt tỉa các vi phân nảy loạn xạ khi model học các mẫu Hard bị drift
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_1.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/30] - Train Loss: 0.8576, Train Macro F1: 0.8835 | Val Loss: 0.6106, Val Macro F1: 0.9077


Epoch [2/30] - Train Loss: 0.6885, Train Macro F1: 0.9590 | Val Loss: 0.6325, Val Macro F1: 0.8978


Epoch [3/30] - Train Loss: 0.6758, Train Macro F1: 0.9646 | Val Loss: 0.6149, Val Macro F1: 0.8967


Epoch [4/30] - Train Loss: 0.6700, Train Macro F1: 0.9677 | Val Loss: 0.5845, Val Macro F1: 0.9093


...

Epoch [19/30] - Train Loss: 0.5984, Train Macro F1: 0.9965 | Val Loss: 0.5827, Val Macro F1: 0.9412


Epoch [20/30] - Train Loss: 0.5986, Train Macro F1: 0.9959 | Val Loss: 0.5801, Val Macro F1: 0.9396


Epoch [21/30] - Train Loss: 0.5976, Train Macro F1: 0.9962 | Val Loss: 0.5822, Val Macro F1: 0.9426


Epoch [22/30] - Train Loss: 0.5946, Train Macro F1: 0.9972 | Val Loss: 0.5819, Val Macro F1: 0.9399

[Early Stopping] Model không cải thiện sau 10 epochs. Dừng huấn luyện.

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---


              precision    recall  f1-score   support

           0     0.8964    0.8044    0.8479     19846
           1     0.9999    1.0000    1.0000    484077
           2     0.3141    0.9920    0.4771      2515
           3     0.9967    0.8237    0.9019     36253
           4     0.5805    0.8342    0.6846     18979
           5     0.9980    0.9971    0.9976      2451
           6     0.6734    0.6942    0.6836     11847
           7     1.0000    1.0000    1.0000    107436
           8     0.6950    0.9952    0.8184     16746
           9     0.9989    0.6711    0.8028     41514
          10     0.9552    0.9858    0.9703     18567

    accuracy                         0.9591    760231
   macro avg     0.8280    0.8907    0.8349    760231
weighted avg     0.9714    0.9591    0.9613    760231



Undersampling + CNN + LSTM + Attention + Sliding Window

In [16]:
import pandas as pd
train_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\test_df_prepared.parquet", engine="pyarrow")
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

train_df.sort_values(by='timestamp_num', inplace=True)
valid_df.sort_values(by='timestamp_num', inplace=True)
test_df.sort_values(by='timestamp_num', inplace=True)

cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [17]:
import torch
from torch.utils.data import Dataset
import numpy as np

class UndersampledSlidingWindowDataset(Dataset):
    def __init__(self, df, time_steps, max_samples_per_class=20000, step=5):
        self.X = torch.tensor(df.drop(columns=['label']).values, dtype=torch.float32)
        self.y = torch.tensor(df['label'].values, dtype=torch.long)
        self.time_steps = time_steps
        self.step = step
        
        print(f"Đang tính toán các cửa sổ hợp lệ (global step={step}) và Undersampling...")
        
        # tạo mảng index cho step = 1 (dành riêng cho các class hiếm cần bảo toàn)
        all_indices_step1 = np.arange(0, len(self.X) - self.time_steps + 1, 1)
        window_labels_step1 = self.y[all_indices_step1 + self.time_steps - 1].numpy()
        
        # tạo mảng index theo step được truyền vào (cho các class còn lại)
        all_indices_stepped = np.arange(0, len(self.X) - self.time_steps + 1, self.step)
        window_labels_stepped = self.y[all_indices_stepped + self.time_steps - 1].numpy()
        
        self.valid_indices = []
        
        # lặp qua từng class (Lấy từ step=1 để đảm bảo không sót class nào)
        classes = np.unique(window_labels_step1)
        rng = np.random.default_rng(seed=42)
        for c in classes:
            # ƯU TIÊN GIỮ NGUYÊN băm cửa sổ 1-1 cho Class 2 và 6
            if c in [100]:
                c_indices = all_indices_step1[window_labels_step1 == c]
            else:
                c_indices = all_indices_stepped[window_labels_stepped == c]
                
            count = len(c_indices)
            
            # nếu class đó có nhiều mẫu hơn ngưỡng max_samples_per_class
            if count > max_samples_per_class:
                # chọn ngẫu nhiên không hoàn lại
                 
                c_indices = rng.choice(c_indices, max_samples_per_class, replace=False)
                print(f"Class {c}: Giảm từ {count} xuống {max_samples_per_class} cửa sổ.")
            else:
                # nếu ít hơn thì giữ nguyên
                if c in [2, 6]:
                    print(f"Class {c}: Giữ nguyên {count} cửa sổ (sử dụng step=1 để bảo toàn).")
                else:
                    print(f"Class {c}: Giữ nguyên {count} cửa sổ (step={self.step}).")
                
            self.valid_indices.extend(c_indices)
            
        # Trộn đều danh sách index để các batch đa dạng hơn
        rng.shuffle(self.valid_indices)
        self.valid_indices = np.array(self.valid_indices) # Chuyển sang ndarray
        print(f"Tổng số cửa sổ sau khi lọc và Undersampling: {len(self.valid_indices)}")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        # lấy các index đã được lọc và xáo trộn
        actual_idx = self.valid_indices[idx]
        
        # lấy feature và label của cửa sổ trượt tại index thực sự
        window_X = self.X[actual_idx : actual_idx + self.time_steps]
        label_y = self.y[actual_idx + self.time_steps - 1]
        
        return window_X, label_y

In [18]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

MAX_SAMPLES = 20000 
TIME_STEPS = 10
BATCH_SIZE = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TEST_STEP_SIZE = 1
TRAIN_STEP_SIZE = 5 
NUM_FEATURES = train_df.shape[1] - 1
NUM_CLASSES = len(train_df['label'].unique())

# tối đa mỗi class sẽ có 20000 của sổ trượt
print(f"Khởi tạo tập Train (có Undersampling, set Train Step = {TRAIN_STEP_SIZE})...")
train_dataset = UndersampledSlidingWindowDataset(train_df, TIME_STEPS, max_samples_per_class=MAX_SAMPLES, step=TRAIN_STEP_SIZE)

print(f"Khởi tạo tập Val và Test (Không Undersampling, giữ nguyên gốc, set Test Step = {TEST_STEP_SIZE})...")
# để max_samples_per_class cho tập val và test lớn để không xóa cửa sổ trượt nào
val_dataset = UndersampledSlidingWindowDataset(valid_df, TIME_STEPS, max_samples_per_class=10000000, step=1) 
test_dataset = UndersampledSlidingWindowDataset(test_df, TIME_STEPS, max_samples_per_class=10000000, step=1)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Khởi tạo tập Train (có Undersampling, set Train Step = 5)...
Đang tính toán các cửa sổ hợp lệ (global step=5) và Undersampling...
Class 0: Giữ nguyên 12897 cửa sổ (step=5).
Class 1: Giảm từ 314644 xuống 20000 cửa sổ.
Class 2: Giữ nguyên 1633 cửa sổ (sử dụng step=1 để bảo toàn).
Class 3: Giảm từ 23563 xuống 20000 cửa sổ.
Class 4: Giữ nguyên 12333 cửa sổ (step=5).
Class 5: Giữ nguyên 1592 cửa sổ (step=5).
Class 6: Giữ nguyên 7698 cửa sổ (sử dụng step=1 để bảo toàn).
Class 7: Giảm từ 69830 xuống 20000 cửa sổ.
Class 8: Giữ nguyên 10884 cửa sổ (step=5).
Class 9: Giảm từ 26987 xuống 20000 cửa sổ.
Class 10: Giữ nguyên 12065 cửa sổ (step=5).
Tổng số cửa sổ sau khi lọc và Undersampling: 139102
Khởi tạo tập Val và Test (Không Undersampling, giữ nguyên gốc, set Test Step = 1)...
Đang tính toán các cửa sổ hợp lệ (global step=1) và Undersampling...
Class 0: Giữ nguyên 14880 cửa sổ (step=1).
Class 1: Giữ nguyên 363049 cửa sổ (step=1).
Class 2: Giữ nguyên 1884 cửa sổ (sử dụng step=1 để bảo toàn).
Cla

In [19]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        # lớp Linear để tính điểm số (score) cho từng time-step
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs):
        # đầu ra bi-lstm: (Batch, SeqLen, Hidden*2)
        
        # tính điểm chú tý cho mỗi timestap => (batch, seqlen, 1)
        scores = self.attention(lstm_outputs)
        
        # áp dung softmax để đưa sequence attention scores về dạng xác suất
        weights = torch.softmax(scores, dim=1)
        
        # nhân trọng số với output của lstm và tỉnh tổng (nhân chập)
        # (Batch, SeqLen, 1) * (Batch, SeqLen, Hidden*2) -> (Batch, Hidden*2)
        context_vector = torch.sum(weights * lstm_outputs, dim=1)
        
        # trả lại context_vector và weights (để phục vụ cho việc trực quan hóa attention sau này)
        return context_vector, weights

# model CNN-BiLSTM với Cơ chế Attention
class CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes, time_steps, hidden_size=128):
        super(CNN_BiLSTM_Attention, self).__init__()
        
        # conv1d có padding=1 => giữ nguyên chiều dài chuỗi
        self.conv1 = nn.Conv1d(in_channels=num_features, out_channels=64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(64)
        self.dropout_cnn = nn.Dropout1d(0.2)

        # conv1d có padding=1 => giữ nguyên chiều dài chuỗi, tăng kênh lên 128
        self.conv2 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(128)
        
        self.relu = nn.ReLU()
        
        # giảm 2 timesteps lại còn 1
        self.pool = nn.MaxPool1d(kernel_size=2) 
        
        # bi-lstm như cũ
        self.bilstm = nn.LSTM(input_size=128, hidden_size=hidden_size, 
                              batch_first=True, bidirectional=True)
        

        # kích thước đầu vào của Attention = hidden_size * 2 (vì BiLSTM ghép 2 chiều ngược xuôi)
        self.attention = Attention(hidden_size * 2)
        
        self.dropout = nn.Dropout(0.5)
        
        # tầng phân loại cuối cùng
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        self.fc2 = nn.Linear(64, num_classes)
        
    def forward(self, x):
        # đầu vào (256,10, 314)
        
        # đảo trục cho CNN -> (256, 314, 10)
        x = x.permute(0, 2, 1) 
        
        # đi qua cnn block 1 => (256, 64, 10)
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout_cnn(x)

        # đi qua cnn block 2 => (256, 128, 10)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        
        # giảm chiều dài chuỗi đi 2 lần => (256, 128, 5)
        x = self.pool(x)
        
        # đảo trục lại cho lstm (256, 5, 128)
        x = x.permute(0, 2, 1)
        
        # đi qua bi-lstm => (256, 5, 256)
        out, (hn, cn) = self.bilstm(x)
        
        # dùng attention để tính ra context vector: (256, 256)
        context_vector, attn_weights = self.attention(out)
        
        # đi qua lớp dense để ra dự đoán nhãn
        out = self.dropout(context_vector)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        
        return out

In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Paper_CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes):
        super(Paper_CNN_BiLSTM_Attention, self).__init__()
        
        # THAY ĐỔI DUY NHẤT Ở ĐÂY: in_channels = num_features (thay vì 1)
        # Conv1d sẽ trượt dọc theo chiều thời gian (Time_Steps) thay vì trượt dọc theo các features.
        self.conv1 = nn.Conv1d(in_channels=num_features, out_channels=128, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(128)
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        self.drop1 = nn.Dropout1d(0.3)

        # tầng 2: giữ nguyên
        self.conv2 = nn.Conv1d(in_channels=128, out_channels=256, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(256)
        self.pool2 = nn.MaxPool1d(kernel_size=2)
        self.drop2 = nn.Dropout1d(0.3)

        # tầng 3: giữ nguyên
        self.conv3 = nn.Conv1d(in_channels=256, out_channels=128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool3 = nn.MaxPool1d(kernel_size=2)
        self.drop3 = nn.Dropout1d(0.3)

        # tầng BiLSTM: giữ nguyên
        self.bilstm = nn.LSTM(input_size=128, hidden_size=128, batch_first=True, bidirectional=True)
        self.drop_lstm = nn.Dropout(0.3)

        # tầng attention: giữ nguyên
        self.attention = Attention(128 * 2) 

        # các tầng dense: giữ nguyên
        self.fc1 = nn.Linear(128 * 2, 256)
        self.drop_fc1 = nn.Dropout(0.4)

        self.fc2 = nn.Linear(256, 128)
        self.drop_fc2 = nn.Dropout(0.4)

        # tầng phân loại cuối cùng: giữ nguyên
        self.fc3 = nn.Linear(128, num_classes)

    def forward(self, x):
        # Đầu vào x từ SlidingWindowDataset đang có shape: (Batch, Time_Steps, Features)
        
        # THAY ĐỔI Ở ĐÂY: Dùng permute thay vì unsqueeze
        # Chuyển vị để shape trở thành: (Batch, Features, Time_Steps)
        # Để Conv1d quét qua chiều Time_Steps với số kênh tương ứng với số Features
        x = x.permute(0, 2, 1) 

        # đi qua 3 tầng Conv1dCNN (Code đoạn này giữ nguyên 100%)
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.pool1(x)
        x = self.drop1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.pool2(x)
        x = self.drop2(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = F.relu(x)
        x = self.pool3(x)
        x = self.drop3(x)

        # permute để đưa qua BiLSTM: (Batch, Channels, Time_Steps_New) -> (Batch, Time_Steps_New, Channels)
        # Chiều Channels lúc này đang là 128 (khớp hoàn hảo với input_size=128 của BiLSTM)
        x = x.permute(0, 2, 1)

        # đi qua BiLSTM
        out, _ = self.bilstm(x)
        out = self.drop_lstm(out)

        # đi qua attention để lấy context vector
        context_vector, _ = self.attention(out)

        # đi qua các lớp dense
        out = self.fc1(context_vector)
        out = F.relu(out)
        out = self.drop_fc1(out)

        out = self.fc2(out)
        out = F.relu(out)
        out = self.drop_fc2(out)

        out = self.fc3(out)
        
        return out

In [21]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score

model = Paper_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES).to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 10

for epoch in range(EPOCHS):
    
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping để cắt tỉa các vi phân nảy loạn xạ khi model học các mẫu Hard bị drift
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_1.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/30] - Train Loss: 0.9966, Train Macro F1: 0.6355 | Val Loss: 0.6474, Val Macro F1: 0.7204


Epoch [2/30] - Train Loss: 0.7332, Train Macro F1: 0.8777 | Val Loss: 0.6317, Val Macro F1: 0.8692


Epoch [3/30] - Train Loss: 0.6958, Train Macro F1: 0.9289 | Val Loss: 0.6471, Val Macro F1: 0.8423


Epoch [4/30] - Train Loss: 0.6825, Train Macro F1: 0.9364 | Val Loss: 0.6300, Val Macro F1: 0.8869


...

Epoch [27/30] - Train Loss: 0.6035, Train Macro F1: 0.9788 | Val Loss: 0.5914, Val Macro F1: 0.9232


Epoch [28/30] - Train Loss: 0.6029, Train Macro F1: 0.9792 | Val Loss: 0.5966, Val Macro F1: 0.9264


Epoch [29/30] - Train Loss: 0.6032, Train Macro F1: 0.9792 | Val Loss: 0.5958, Val Macro F1: 0.9195


Epoch [30/30] - Train Loss: 0.6042, Train Macro F1: 0.9789 | Val Loss: 0.5944, Val Macro F1: 0.9201

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---


              precision    recall  f1-score   support

           0     0.7382    0.8005    0.7681     19846
           1     0.9999    1.0000    1.0000    484077
           2     0.3647    0.9980    0.5342      2515
           3     0.9997    0.8732    0.9322     36253
           4     0.4723    0.6921    0.5615     18979
           5     0.9988    0.9951    0.9969      2451
           6     0.8996    0.7014    0.7883     11847
           7     1.0000    1.0000    1.0000    107436
           8     0.6967    0.9504    0.8040     16746
           9     1.0000    0.6667    0.8000     41514
          10     0.9882    0.9893    0.9888     18567

    accuracy                         0.9568    760231
   macro avg     0.8325    0.8788    0.8340    760231
weighted avg     0.9693    0.9568    0.9594    760231



Undersampling + CNN + LSTM + Attention (attention + no sliding window) (cấu trúc giống paper gốc)

In [22]:
import pandas as pd
train_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\test_df_prepared.parquet", engine="pyarrow")
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

train_df.sort_values(by='timestamp_num', inplace=True)
valid_df.sort_values(by='timestamp_num', inplace=True)
test_df.sort_values(by='timestamp_num', inplace=True)

cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F
NUM_FEATURES = train_df.shape[1] - 1 # trừ đi cột label
NUM_CLASSES = len(train_df['label'].unique()) # số class
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs):
        # tính điểm chú ý
        scores = self.attention(lstm_outputs)
        weights = torch.softmax(scores, dim=1)
        # Nhân trọng số và tính tổng
        context_vector = torch.sum(weights * lstm_outputs, dim=1)
        return context_vector, weights

# kiến trúc chính của paper
class Paper_CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes):
        super(Paper_CNN_BiLSTM_Attention, self).__init__()
        
        # tầng 1: conv1d với 128 filters, kernel size = 5, padding = 2 để giũ nguyên chiều dài chuỗi
        # in_channels = 1 (Vì ta coi 314 features như 1 chuỗi dài 314 đơn vị) (đưa từng flow riêng lẻ vào)
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=128, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(128)
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        self.drop1 = nn.Dropout1d(0.3)

        # tầng 2: conv1d với 256 filters, kernel size 5, ReLU -> Batch Norm, MaxPool(2), Dropout(0.3)
        self.conv2 = nn.Conv1d(in_channels=128, out_channels=256, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(256)
        self.pool2 = nn.MaxPool1d(kernel_size=2)
        self.drop2 = nn.Dropout1d(0.3)

        # tầng 3: conv1d với 128 filters, kernel size 3, ReLU -> Batch Norm, MaxPool(2), Dropout(0.3)
        self.conv3 = nn.Conv1d(in_channels=256, out_channels=128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool3 = nn.MaxPool1d(kernel_size=2)
        self.drop3 = nn.Dropout1d(0.3)

        # tầng BiLSTM: 128 units, Dropout(0.3)
        self.bilstm = nn.LSTM(input_size=128, hidden_size=128, batch_first=True, bidirectional=True)
        self.drop_lstm = nn.Dropout(0.3)

        # tầng attention
        self.attention = Attention(128 * 2) # *2 vì là Bidirectional

        # các tầng dense
        self.fc1 = nn.Linear(128 * 2, 256)
        self.drop_fc1 = nn.Dropout(0.4)

        self.fc2 = nn.Linear(256, 128)
        self.drop_fc2 = nn.Dropout(0.4)

        # tầng phân loại cuối cùng
        self.fc3 = nn.Linear(128, num_classes)

    def forward(self, x):
        # Đầu vào x từ StandardFlowDataset đang có shape: (Batch, Features)
        # đầu vào x có shape: (256, 314)
        
        # shape trở thành: (256, 1, 314)
        x = x.unsqueeze(1) 

        # đi qua 3 tầng Conv1dCNN
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.pool1(x)
        x = self.drop1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.pool2(x)
        x = self.drop2(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = F.relu(x)
        x = self.pool3(x)
        x = self.drop3(x)

        # permute để đưa qua BiLSTM
        x = x.permute(0, 2, 1)

        # đi qua BiLSTM
        out, _ = self.bilstm(x)
        out = self.drop_lstm(out)

        # đi qua attention để lấy context vector
        context_vector, _ = self.attention(out)

        # đi qua các lớp dense
        out = self.fc1(context_vector)
        out = F.relu(out)
        out = self.drop_fc1(out)

        out = self.fc2(out)
        out = F.relu(out)
        out = self.drop_fc2(out)

        out = self.fc3(out)
        
        # không dùng softmax
        return out

# khởi tạo mô hình và đẩy lên GPU
model = Paper_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES).to(DEVICE)
print(model)

Paper_CNN_BiLSTM_Attention(
  (conv1): Conv1d(1, 128, kernel_size=(5,), stride=(1,), padding=(2,))
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool1): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (drop1): Dropout1d(p=0.3, inplace=False)
  (conv2): Conv1d(128, 256, kernel_size=(5,), stride=(1,), padding=(2,))
  (bn2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (drop2): Dropout1d(p=0.3, inplace=False)
  (conv3): Conv1d(256, 128, kernel_size=(3,), stride=(1,), padding=(1,))
  (bn3): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (drop3): Dropout1d(p=0.3, inplace=False)
  (bilstm): LSTM(128, 128, batch_first=True, bidirectional=True)
  (drop_lstm): Dropout(p=0.3, inp

In [24]:
from torch.utils.data import Dataset
import numpy as np
class StandardFlowDataset(Dataset):
    def __init__(self, df, max_samples_per_class=None):
        self.X = []
        self.y = []
        
        # nhóm theo nhãn và áp dụng undersampling
        for class_name, group in df.groupby('label'):
            if max_samples_per_class and len(group) > max_samples_per_class:
                group = group.sample(n=max_samples_per_class, random_state=42)
            
            self.X.append(group.drop(columns=['label']).values)
            self.y.append(group['label'].values)
            
        self.X = np.vstack(self.X)
        self.y = np.concatenate(self.y)
        
        # xáo trộn dữ liệu sau khi đã áp dụng undersampling để đảm bảo tính ngẫu nhiên
        idx = np.random.permutation(len(self.X))
        self.X = torch.tensor(self.X[idx], dtype=torch.float32)
        self.y = torch.tensor(self.y[idx], dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# khởi tạo dataset với max_samples_per_class để giới hạn số lượng mẫu của mỗi class trong tập train, còn val và test thì giữ nguyên để đánh giá chính xác hơn
MAX_SAMPLES = 20000 
train_dataset = StandardFlowDataset(train_df, max_samples_per_class=MAX_SAMPLES)
val_dataset = StandardFlowDataset(valid_df)
test_dataset = StandardFlowDataset(test_df) 

BATCH_SIZE = 256
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [25]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score

model = Paper_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES).to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 10

for epoch in range(EPOCHS):
    
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping để cắt tỉa các vi phân nảy loạn xạ khi model học các mẫu Hard bị drift
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_1.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/30] - Train Loss: 1.4334, Train Macro F1: 0.5397 | Val Loss: 0.7092, Val Macro F1: 0.6877


Epoch [2/30] - Train Loss: 1.0923, Train Macro F1: 0.7284 | Val Loss: 0.7014, Val Macro F1: 0.7173


Epoch [3/30] - Train Loss: 1.0420, Train Macro F1: 0.7507 | Val Loss: 0.6842, Val Macro F1: 0.7133


Epoch [4/30] - Train Loss: 1.0091, Train Macro F1: 0.7663 | Val Loss: 0.6825, Val Macro F1: 0.7285


...

Epoch [27/30] - Train Loss: 0.8745, Train Macro F1: 0.8454 | Val Loss: 0.6356, Val Macro F1: 0.8373


Epoch [28/30] - Train Loss: 0.8758, Train Macro F1: 0.8455 | Val Loss: 0.6411, Val Macro F1: 0.8164


Epoch [29/30] - Train Loss: 0.8773, Train Macro F1: 0.8455 | Val Loss: 0.6386, Val Macro F1: 0.8299


Epoch [30/30] - Train Loss: 0.8577, Train Macro F1: 0.8544 | Val Loss: 0.6329, Val Macro F1: 0.8316

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---


              precision    recall  f1-score   support

           0     0.6875    0.7032    0.6953     19846
           1     1.0000    0.9999    0.9999    484077
           2     0.4690    0.7781    0.5852      2515
           3     0.9784    0.8963    0.9355     36253
           4     0.6484    0.8288    0.7276     18979
           5     0.9463    0.7336    0.8265      2451
           6     0.5548    0.7132    0.6241     11847
           7     0.9995    0.9963    0.9979    107436
           8     0.9104    0.9033    0.9068     16746
           9     0.9976    0.6940    0.8185     41523
          10     0.5595    0.7399    0.6372     18567

    accuracy                         0.9512    760240
   macro avg     0.7956    0.8169    0.7959    760240
weighted avg     0.9602    0.9512    0.9532    760240



Hybrid Sampling (SMOTE + Undersampling) + CNN + LSTM + Attention + No Sliding Window

In [26]:
import pandas as pd
train_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\test_df_prepared.parquet", engine="pyarrow")
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

train_df.sort_values(by='timestamp_num', inplace=True)
valid_df.sort_values(by='timestamp_num', inplace=True)
test_df.sort_values(by='timestamp_num', inplace=True)

cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [ ]:
from imblearn.over_sampling import BorderlineSMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
from imblearn.combine import SMOTETomek
y_train = train_df['label']
x_train = train_df.drop(columns=['label'])
TARGET_COUNT = 100000
counts = y_train.value_counts().to_dict()



under_strategy = {label: TARGET_COUNT for label, count in counts.items() if count > TARGET_COUNT}
over_strategy = {label: TARGET_COUNT for label, count in counts.items() if count < TARGET_COUNT}

under = RandomUnderSampler(sampling_strategy=under_strategy, random_state=42)
smote_tomek = SMOTETomek(sampling_strategy=over_strategy, random_state=42, n_jobs=-1)

pipeline = Pipeline(steps=[('under', under), ('over', smote_tomek)])

print("Start to apply Hybrid Sampling (Under + Borderline SMOTE)...")
x_train_resampled, y_train_resampled = pipeline.fit_resample(x_train, y_train)

print(f"Number of samples after resampling: {len(y_train_resampled)}")
print(y_train_resampled.value_counts())
train_resampled_df = pd.DataFrame(x_train_resampled, columns=x_train.columns)
train_resampled_df['label'] = y_train_resampled

Start to apply Hybrid Sampling (Under + Borderline SMOTE)...


In [ ]:
from torch.utils.data import Dataset
import torch
import numpy as np
from torch.utils.data import DataLoader
class StandardFlowDataset(Dataset):
    def __init__(self, df, max_samples_per_class=None):
        self.X = []
        self.y = []
        
        for class_name, group in df.groupby('label'):
            if max_samples_per_class and len(group) > max_samples_per_class:
                group = group.sample(n=max_samples_per_class, random_state=42)
            
            self.X.append(group.drop(columns=['label']).values)
            self.y.append(group['label'].values)
            
        self.X = np.vstack(self.X)
        self.y = np.concatenate(self.y)
        
        idx = np.random.permutation(len(self.X))
        self.X = torch.tensor(self.X[idx], dtype=torch.float32)
        self.y = torch.tensor(self.y[idx], dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
BATCH_SIZE = 512
train_dataset = StandardFlowDataset(train_resampled_df)
val_dataset = StandardFlowDataset(valid_df)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

test_dataset = StandardFlowDataset(test_df)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
NUM_FEATURES = train_dataset.X.shape[1]
NUM_CLASSES = len(train_dataset.y.unique())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs):
        scores = self.attention(lstm_outputs)
        weights = torch.softmax(scores, dim=1)
        context_vector = torch.sum(weights * lstm_outputs, dim=1)
        return context_vector, weights

class Paper_CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes):
        super(Paper_CNN_BiLSTM_Attention, self).__init__()
        
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=128, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(128)
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        self.drop1 = nn.Dropout1d(0.3)

        self.conv2 = nn.Conv1d(in_channels=128, out_channels=256, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(256)
        self.pool2 = nn.MaxPool1d(kernel_size=2)
        self.drop2 = nn.Dropout1d(0.3)

        self.conv3 = nn.Conv1d(in_channels=256, out_channels=128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool3 = nn.MaxPool1d(kernel_size=2)
        self.drop3 = nn.Dropout1d(0.3)

        self.bilstm = nn.LSTM(input_size=128, hidden_size=128, batch_first=True, bidirectional=True)
        self.drop_lstm = nn.Dropout(0.3)

        self.attention = Attention(128 * 2) 

        self.fc1 = nn.Linear(128 * 2, 256)
        self.drop_fc1 = nn.Dropout(0.4)

        self.fc2 = nn.Linear(256, 128)
        self.drop_fc2 = nn.Dropout(0.4)

        self.fc3 = nn.Linear(128, num_classes)

    def forward(self, x):
        
        x = x.unsqueeze(1) 

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.pool1(x)
        x = self.drop1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.pool2(x)
        x = self.drop2(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = F.relu(x)
        x = self.pool3(x)
        x = self.drop3(x)

        x = x.permute(0, 2, 1)

        out, _ = self.bilstm(x)
        out = self.drop_lstm(out)

        context_vector, _ = self.attention(out)

        out = self.fc1(context_vector)
        out = F.relu(out)
        out = self.drop_fc1(out)

        out = self.fc2(out)
        out = F.relu(out)
        out = self.drop_fc2(out)

        out = self.fc3(out)
        
        return out


In [ ]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score
import torch.optim as optim

model = Paper_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES).to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 10

for epoch in range(EPOCHS):
    
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping để cắt tỉa các vi phân nảy loạn xạ khi model học các mẫu Hard bị drift
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_1.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/30] - Train Loss: 1.1169, Train Macro F1: 0.6881 | Val Loss: 0.6937, Val Macro F1: 0.7352


Epoch [2/30] - Train Loss: 0.9283, Train Macro F1: 0.7824 | Val Loss: 0.6754, Val Macro F1: 0.7546


Epoch [3/30] - Train Loss: 0.8822, Train Macro F1: 0.8083 | Val Loss: 0.6656, Val Macro F1: 0.7625


Epoch [4/30] - Train Loss: 0.8654, Train Macro F1: 0.8177 | Val Loss: 0.6567, Val Macro F1: 0.7793


...

Epoch [27/30] - Train Loss: 0.8054, Train Macro F1: 0.8781 | Val Loss: 0.6444, Val Macro F1: 0.7871


Epoch [28/30] - Train Loss: 0.8044, Train Macro F1: 0.8786 | Val Loss: 0.6558, Val Macro F1: 0.7708


Epoch [29/30] - Train Loss: 0.8058, Train Macro F1: 0.8784 | Val Loss: 0.6435, Val Macro F1: 0.7833


Epoch [30/30] - Train Loss: 0.7951, Train Macro F1: 0.8836 | Val Loss: 0.6366, Val Macro F1: 0.8132

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---


              precision    recall  f1-score   support

           0     0.7880    0.6806    0.7304     19846
           1     1.0000    0.9999    0.9999    484077
           2     0.4146    0.9571    0.5785      2515
           3     0.9903    0.8573    0.9190     36253
           4     0.6265    0.8335    0.7154     18979
           5     0.6831    0.9992    0.8115      2451
           6     0.5960    0.7157    0.6504     11847
           7     1.0000    0.9963    0.9982    107436
           8     0.9727    0.9074    0.9389     16746
           9     1.0000    0.6653    0.7990     41523
          10     0.5373    0.8257    0.6510     18567

    accuracy                         0.9510    760240
   macro avg     0.7826    0.8580    0.7993    760240
weighted avg     0.9635    0.9510    0.9534    760240

